# **Homework 2 - Image Captioning**

<div style="border: 3px solid #222; padding: 16px; border-radius: 10px; background-color: #1c1f26; font-family: 'Helvetica Neue', Helvetica, Arial, sans-serif; color: #e0e0e0;">
  <div style="display: flex; align-items: center; gap: 8px; margin-top: 12px;">
    <span style="font-size: 24px; color: #ff5555;">&#128274;</span>
    <span style="font-size: 16px;"><strong>Project:</strong> Homeworks</span>
  </div>
  <div style="display: flex; align-items: center; gap: 8px; margin-top: 12px;">
    <span style="font-size: 24px; color: #ff5555;">&#128218;</span>
    <span style="font-size: 16px;"><strong>Course:</strong> Deep Network Development 25/26/2</span>
  </div>
  <div style="display: flex; align-items: center; gap: 8px; margin-top: 12px;">
    <span style="font-size: 24px; color: #6e8192;">&#128100;</span>
    <span style="font-size: 16px;"><strong>Authors:</strong> Tamás Takács (PhD student, Department of Artificial Intelligence, Eötvös Loránd University) </span>
  </div>
</div>
<hr style="border: none; border-top: 2px solid #444;">
<br>

<img src="https://repository-images.githubusercontent.com/83958320/8f162500-8ace-11e9-94ee-0b86d27bbc5e" alt="1" border="0">

This notebook contains the required task for the **seccond homework** of the **Deep Network Development (DNDEG)** course. Read the task description carefully and **fill in the empty code cells**.

---

## **Task Description**

Train your **own custom image captioning model** and compare its performance with an existing **pre-trained** model using the `MS COCO Captions 2017` dataset.

### Architecture

Your model must follow an **Encoder-Attention-Transformer Decoder** architecture:
- **Encoder:** Pre-trained CNN backbone (e.g., ResNet) that must be **fine-tuned** during training.
- **Decoder:** Transformer-based with multi-head self-attention and cross-attention layers.
- **Controllable Generation:** At least one control mechanism (caption length, style, or focus).
- **Cross-Attention Visualization:** Visualize which image regions the decoder attends to for each generated word.

### Dataset

The **MS COCO Captions 2017** dataset:
- **Training set:** ~118K images with 5 captions each
- **Validation set:** ~5K images with 5 captions each

**Download links:**
- Images (Train): http://images.cocodataset.org/zips/train2017.zip (~18GB)
- Images (Val): http://images.cocodataset.org/zips/val2017.zip (~1GB)
- Annotations: http://images.cocodataset.org/annotations/annotations_trainval2017.zip (~241MB)

Consider limiting the vocabulary to words appearing at least 5 times.

> ⚠️ **Hardware Note:** You may reduce the dataset size (e.g., use a subset of the training images) to fit your current hardware capabilities or cloud resource availability. Ensure you document any such modifications in your notebook.

---

## **Requirements**

**Data Preparation:**
- Download and prepare the MS COCO Captions 2017 dataset
- Display sample images with their original and tokenized captions

**Model Training:**
- Train an Encoder-Attention-Transformer Decoder model
- Track training and validation loss with visualizations
- Monitor BLEU-1, BLEU-2, BLEU-3, and BLEU-4 scores
- Implement overfitting prevention (early stopping, regularization, learning rate scheduling)
- Save the best-performing model checkpoint

**Evaluation:**
- Compute BLEU scores using all 5 reference captions per image
- Visualize cross-attention weights showing what the model attends to
- Compare your model with a pre-trained model (e.g., BLIP, ViT-GPT2) on the same validation images
- Provide written analysis of the strengths and weaknesses of both models

> ⚠️ **Performance Note:** Your model is not expected to achieve state-of-the-art results, but it should perform better than random guessing and show decreasing loss during training.

## **Notebook Structure**

The following sections will guide you through the task:

0. Necessary imports
1. Data loading process
2. Defining data augmentations
3. Creating datasets and dataloaders
4. Visualizing training data
5. Creating the image captioning model
6. Defining loss function and optimizer
7. Training the model
8. Evaluation (metrics, cross-attention visualization, inference)
9. Loading a pre-trained model
10. Evaluating the pre-trained model
11. Comparing the two models

The sections are there to guide you but you **do not have to follow them strictly**. Feel free to add more code cells as needed, but keep all code within this notebook (no external `.py` files).

---

## **Submission**

- Upload the final `.ipynb` file to [Canvas](https://canvas.elte.hu) before the deadline (check Canvas for the exact date)
- Copying others' code results in automatic failure
- Please add your **name** and **Neptun ID** in the cell below

---

## **Grading and Feedback**

The submitted notebook is graded as **accepted** or **not accepted**. There is no detailed scoring on the notebook itself. You will receive **written feedback** on your submission within 3 days of the final deadline (earlier if you submit early).

Your grade for this homework is determined at the **oral exam**:
- **15 points**: theoretical questions on your homework (minimum **6/15** required)
- **5 points** if your homework is accepted

Your homework **must be accepted** in order to take the oral exam.

> ⚠️ **Hard requirement:** The assignment will be marked **not accepted** if the training has not been run for **at least 5 epoch**. Ensure your notebook contains evidence of completed training (loss values, saved checkpoints, etc.).

<div style="border: 3px dashed #ff5555; padding: 16px; border-radius: 10px; background-color: #2a1f1f; font-family: 'Helvetica Neue', Helvetica, Arial, sans-serif; color: #e0e0e0;">
  <div style="font-size: 18px; font-weight: bold; color: #ff5555; margin-bottom: 12px;">
    ✍️ Fill in your details before submitting
  </div>
</div>



**Name:**  Wai Wai Lwin


**Neptun ID:**. N7 COFE

## **0. Necessary Imports**
Import all the necessary packages for this assignment. **ONLY PYTORCH MODELS ARE ACCEPTED!**

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import nltk
import torchvision
import matplotlib.pyplot as plt
import numpy as np
import random
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms
from collections import Counter

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# For COCO dataset loading
# !pip install pycocotools
# from pycocotools.coco import COCO

## **1. Data Loading Process**

For this assignment you will be using the [MS COCO Captions 2017](https://cocodataset.org/#captions-2017) dataset, which contains **captions/descriptions** of diverse images.

<img src="https://cocodataset.org/images/coco-examples.jpg" alt="COCO Examples" border="0" width="600">

**Download Commands:**

```bash
# Download images
wget http://images.cocodataset.org/zips/train2017.zip        # ~18GB, 118K images
wget http://images.cocodataset.org/zips/val2017.zip          # ~1GB, 5K images

# Download annotations (includes captions)
wget http://images.cocodataset.org/annotations/annotations_trainval2017.zip  # ~241MB
```

After downloading, extract and organize the data:

```
coco/
├── images/
│   ├── train2017/
│   └── val2017/
└── annotations/
    ├── captions_train2017.json
    └── captions_val2017.json
```

You can use the official `pycocotools` library to load and parse the annotations:

```python
!pip install pycocotools
from pycocotools.coco import COCO

coco = COCO('annotations/captions_train2017.json')
```

In [ ]:
# ADD YOUR CODE HERE

import os
import zipfile
from pathlib import Path
from pycocotools.coco import COCO

# File Path creation to creat folder
PROJECT_ROOT = Path.cwd() / "coco"
IMAGES = PROJECT_ROOT / "images"
ANNOTATIONS = PROJECT_ROOT / "annotations"

IMAGES.mkdir(parents=True, exist_ok=True)
ANNOTATIONS.mkdir(parents=True, exist_ok=True)

# Download
!wget http://images.cocodataset.org/zips/train2017.zip -P {PROJECT_ROOT}
!wget http://images.cocodataset.org/zips/val2017.zip -P {PROJECT_ROOT}
!wget http://images.cocodataset.org/annotations/annotations_trainval2017.zip -P {PROJECT_ROOT}

# Extract images
with zipfile.ZipFile(PROJECT_ROOT / 'train2017.zip', 'r') as zip_ref:
    zip_ref.extractall(IMAGES)

with zipfile.ZipFile(PROJECT_ROOT / 'val2017.zip', 'r') as zip_ref:
    zip_ref.extractall(IMAGES)

# Extract annotations
with zipfile.ZipFile(PROJECT_ROOT / 'annotations_trainval2017.zip', 'r') as zip_ref:
    zip_ref.extractall(PROJECT_ROOT)

# FIX: Move json files from nested annotations folder
nested = PROJECT_ROOT / 'annotations' / 'annotations'
if nested.exists():
    for file in nested.glob('*.json'):
        file.rename(ANNOTATIONS / file.name)
    nested.rmdir()

# Install pycocotools
!pip install -q pycocotools

# Load captions
coco_train = COCO(ANNOTATIONS / 'captions_train2017.json')
coco_val = COCO(ANNOTATIONS / 'captions_val2017.json')

print("Train:", len(coco_train.imgs))
print("Val:", len(coco_val.imgs))

## **2. Defining Augmentations**

When applying **augmentations** to the `COCO Captions` dataset, it is important to note that these transformations should be applied **only to the images** and not to the captions.

Ensure that your **data augmentation pipeline** includes:
- A **normalization step** to scale pixel values appropriately.
- A **tensor conversion step** to transform images into tensors for model compatibility.
- Additional **augmentations of your choice**, such as random cropping, flipping, or color jittering, to enhance model generalization.

```python
train_transforms = transforms.Compose([
            # Add Augmentations
])

test_transforms = transforms.Compose([
            # Add Augmentations
])
```

In [ ]:
# ADD YOUR CODE HERE


import torchvision.transforms as transforms

# Train transforms - data augmentation
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Test transforms
test_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

## **3. Creating Datasets and Dataloaders**

To load the **COCO Captions** dataset, you need to create a **custom PyTorch** `Dataset` class that returns **images and their corresponding captions**. The captions should be **tokenized** before being returned.

Make sure to include special tokens in your tokenized captions:
- `<sos>` or `<bos>` (Start of Sentence)
- `<eos>` (End of Sentence)
- `<unk>` (Unknown Token) for words outside the vocabulary
- `<pad>` (Padding Token) for batch processing

It is recommended to build a **Vocabulary class** to store all the words in your dataset, as your model can only generate words that exist in this vocabulary. However, saving every word is unnecessary. A **common practice** is to include only words that appear **at least 5 times** across the entire dataset to reduce noise and improve model efficiency.

For the **DataLoader**, ensure that the **batch size** is appropriate so that it fits into memory. Set the **`shuffle`** parameter as follows:

- **Training DataLoader:** `shuffle=True` (to randomize the order of samples)  
- **Validation DataLoader:** `shuffle=False` (to maintain consistency in evaluation)

> **Note**: Remember that batches contain examples with **different caption lengths**. Use a custom `collate_fn` to handle padding and create attention masks for the Transformer decoder.

```python
class Vocabulary:
    """Vocabulary class to map words to indices."""
    def __init__(self, freq_threshold=5):
        raise NotImplementedError

    def build_vocabulary(self, captions):
        raise NotImplementedError

    def numericalize(self, text):
        # Returns: list of token indices
        raise NotImplementedError


class COCOCaptionsDataset(Dataset):
    def __init__(self, root, annFile, vocab, transform=None):
        raise NotImplementedError

    def __len__(self):
        raise NotImplementedError

    def __getitem__(self, idx):
        # Returns: image, caption_tokens, caption_length
        raise NotImplementedError


def collate_fn(batch):
    # Returns: images, padded_captions, lengths, attention_masks
    raise NotImplementedError
```

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import nltk
from pycocotools.coco import COCO
from PIL import Image

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# VOCABULARY CLASS - maps words to indices
class Vocabulary:
    def __init__(self, freq_threshold=5):
        # Special tokens
        self.freq_threshold = freq_threshold
        self.itos = {0: "<pad>", 1: "<sos>", 2: "<eos>", 3: "<unk>"}
        self.stoi = {"<pad>": 0, "<sos>": 1, "<eos>": 2, "<unk>": 3}

    def build_vocabulary(self, captions):
        # Collect all words from captions
        all_words = []
        for caption in captions:
            words = nltk.word_tokenize(caption.lower())
            all_words.extend(words)

        # Count word frequency
        word_counts = Counter(all_words)

        # Add words that appear at least freq_threshold times
        for word, count in word_counts.items():
            if count >= self.freq_threshold and word not in self.stoi:
                idx = len(self.itos)
                self.stoi[word] = idx
                self.itos[idx] = word

    def numericalize(self, text):
        # Convert text to token indices
        words = nltk.word_tokenize(text.lower())
        return [self.stoi.get(word, self.stoi["<unk>"]) for word in words]


# DATASET CLASS - loads images and captions
class COCOCaptionsDataset(Dataset):
    def __init__(self, root, annFile, vocab, transform=None):
        self.root = root
        self.coco = COCO(annFile)  # Load COCO annotations
        self.img_ids = list(self.coco.imgs.keys())  # Get all image IDs
        self.vocab = vocab
        self.transform = transform

        # Store all captions for vocabulary building
        self.all_captions = []
        for img_id in self.img_ids:
            ann_ids = self.coco.getAnnIds(imgIds=img_id)
            anns = self.coco.loadAnns(ann_ids)
            for ann in anns:
                self.all_captions.append(ann['caption'])

    def __len__(self):
        # Return total number of images
        return len(self.img_ids)

    def __getitem__(self, idx):
        # Load image
        img_id = self.img_ids[idx]
        img_info = self.coco.loadImgs(img_id)[0]
        img_path = f"{self.root}/{img_info['file_name']}"
        image = Image.open(img_path).convert('RGB')

        # Apply transforms
        if self.transform:
            image = self.transform(image)

        # Load caption (pick first caption for this image)
        ann_ids = self.coco.getAnnIds(imgIds=img_id)
        anns = self.coco.loadAnns(ann_ids)
        caption = anns[0]['caption']

        # Tokenize caption with <sos> and <eos>
        tokens = self.vocab.numericalize(caption)
        caption_tokens = [self.vocab.stoi["<sos>"]] + tokens + [self.vocab.stoi["<eos>"]]

        return image, torch.tensor(caption_tokens), len(caption_tokens)


# COLLATE FUNCTION - handles padding for batches
def collate_fn(batch):
    images = []
    captions = []
    lengths = []

    # Separate batch items
    for img, cap, length in batch:
        images.append(img)
        captions.append(cap)
        lengths.append(length)

    # Stack images into a batch
    images = torch.stack(images, 0)

    # Pad captions to same length
    max_len = max(lengths)
    padded_captions = torch.zeros(len(batch), max_len, dtype=torch.long)
    attention_masks = torch.zeros(len(batch), max_len, dtype=torch.long)

    # Fill padded tensors
    for i, (cap, length) in enumerate(zip(captions, lengths)):
        padded_captions[i, :length] = cap
        attention_masks[i, :length] = 1  # 1 = real token, 0 = padding

    return images, padded_captions, lengths, attention_masks

## **4. Visualizing Training Data**

To visualize the training data, extract a batch from the training `DataLoader` and plot the **input-target** pairs using `Matplotlib` or `Seaborn`. Ensure that at least **8 pairs** are displayed for a clear representation.

Make sure to **visualize the original and the tokenized caption** as well!

```python
def visualize_batch(dataloader, vocab, num_samples=8):
    raise NotImplementedError
```

In [ ]:
# ADD YOUR CODE HERE


import matplotlib.pyplot as plt
import numpy as np

def visualize_batch(dataloader, vocab, num_samples=8):
    images, padded_captions, lengths, attention_masks = next(iter(dataloader))

    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()

    for i in range(min(num_samples, len(images))):
        # Denormalize image
        img = images[i].permute(1, 2, 0).numpy()
        mean = np.array([0.485, 0.456, 0.406])
        std  = np.array([0.229, 0.224, 0.225])
        img  = np.clip(std * img + mean, 0, 1)

        # Decode tokens to words (skip <pad>, <sos>, <eos>)
        caption_tokens   = padded_captions[i][:lengths[i]].tolist()
        words            = [vocab.itos[idx] for idx in caption_tokens if idx not in [0, 1, 2]]
        original_caption = ' '.join(words)
        tokenized        = [vocab.itos[idx] for idx in caption_tokens]

        # Plot
        axes[i].imshow(img)
        axes[i].axis('off')
        title = original_caption[:50] + "..." if len(original_caption) > 50 else original_caption
        axes[i].set_title(title, fontsize=7)

        print(f"\n--- Sample {i+1} ---")
        print(f"Original : {original_caption}")
        print(f"Tokenized: {tokenized}")
        print(f"Length   : {lengths[i]} tokens")

    plt.tight_layout()
    plt.show()

from pathlib import Path
PROJECT_ROOT = Path.cwd() / "coco"
ANNOTATIONS = PROJECT_ROOT / "annotations"
IMAGES = PROJECT_ROOT / "images"

# Create vocabulary
vocab = Vocabulary(freq_threshold=5)
train_dataset_for_vocab = COCOCaptionsDataset(root=str(IMAGES/'train2017'), annFile=str(ANNOTATIONS/'captions_train2017.json'), vocab=vocab)
vocab.build_vocabulary([ann['caption'] for img_id in train_dataset_for_vocab.img_ids for ann in train_dataset_for_vocab.coco.loadAnns(train_dataset_for_vocab.coco.getAnnIds(imgIds=img_id))])

# Create datasets
train_dataset = COCOCaptionsDataset(root=str(IMAGES/'train2017'), annFile=str(ANNOTATIONS/'captions_train2017.json'), vocab=vocab, transform=train_transforms)
val_dataset = COCOCaptionsDataset(root=str(IMAGES/'val2017'), annFile=str(ANNOTATIONS/'captions_val2017.json'), vocab=vocab, transform=test_transforms)

# Create dataloaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

visualize_batch(train_loader, vocab, num_samples=8)


## **5. Creating the Image Captioning Model**

For this assignment, **you are required to create your own custom image captioning model** and **compare its performance** with an existing pre-trained model.

---

### **Encoder-Attention-Transformer Decoder Architecture**

Your model must follow an **Encoder-Attention-Transformer Decoder** architecture:

#### **1. Encoder (CNN-based)**
The encoder processes images to extract spatial features:
- Use a **pre-trained CNN** (e.g., ResNet-50, ResNet-101) as the backbone.
- Remove the final classification layers to obtain feature maps.
- **Fine-tuning is required** - at minimum, fine-tune the last few layers.
- Output shape should be `(batch_size, num_patches, encoder_dim)` where patches come from spatial locations in the feature map.

#### **2. Transformer Decoder**
The decoder must be **Transformer-based** with the following components:
- **Positional Encoding** for caption token positions.
- **Masked Multi-Head Self-Attention** to prevent attending to future tokens.
- **Cross-Attention** layers to attend to encoder features (image patches).
- **Feed-Forward Networks** with residual connections and layer normalization.

> **Note:** An LSTM/GRU decoder is **not acceptable** for this assignment. You must implement a Transformer decoder.

#### **3. Controllable Generation**
Implement at least one form of caption control:
- **Length Control:** Condition the decoder on desired caption length (short/medium/long).
- **Style Control:** Add style embeddings (formal/casual/descriptive).
- **Focus Control:** Emphasize specific detected objects or regions.

This can be done by adding **control embeddings** to the decoder input or using **special control tokens**.

---

### **Code Skeleton**

```python
class CNNEncoder(nn.Module):
    """CNN-based image encoder using pre-trained backbone."""
    def __init__(self, encoder_dim=2048, fine_tune_from=6):
        super(CNNEncoder, self).__init__()
        # Load pre-trained ResNet, remove final layers
        # Enable fine-tuning for layers >= fine_tune_from
        raise NotImplementedError

    def forward(self, images):
        # Returns: (batch_size, num_patches, encoder_dim)
        raise NotImplementedError
```

```python
class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding for Transformer."""
    def __init__(self, d_model, max_len=100, dropout=0.1):
        super(PositionalEncoding, self).__init__()
        raise NotImplementedError

    def forward(self, x):
        raise NotImplementedError
```

```python
class TransformerDecoderLayer(nn.Module):
    """Single Transformer decoder layer with masked self-attention and cross-attention."""
    def __init__(self, d_model, num_heads, dim_feedforward, dropout=0.1):
        super(TransformerDecoderLayer, self).__init__()
        # Masked multi-head self-attention
        # Cross-attention to encoder features
        # Feed-forward network
        raise NotImplementedError

    def forward(self, tgt, memory, tgt_mask=None, tgt_key_padding_mask=None):
        # Returns: output, cross_attention_weights (for visualization)
        raise NotImplementedError
```

```python
class TransformerDecoder(nn.Module):
    """Full Transformer decoder with embedding and output projection."""
    def __init__(self, vocab_size, d_model, num_heads, num_layers, dim_feedforward, dropout=0.1):
        super(TransformerDecoder, self).__init__()
        # Word embedding + positional encoding
        # Control embedding (for controllable generation)
        # Stack of TransformerDecoderLayer
        # Output projection to vocab
        raise NotImplementedError

    def forward(self, captions, encoder_features, tgt_mask=None, control_signal=None):
        # control_signal: optional (e.g., desired length, style)
        # Returns: logits, list of cross_attention_weights from each layer
        raise NotImplementedError

    def generate_square_subsequent_mask(self, sz):
        """Generate causal mask for autoregressive decoding."""
        raise NotImplementedError
```

```python
class ImageCaptioningModel(nn.Module):
    """Full image captioning model: Encoder + Transformer Decoder."""
    def __init__(self, vocab_size, encoder_dim, d_model, num_heads, num_layers, dim_feedforward):
        super(ImageCaptioningModel, self).__init__()
        raise NotImplementedError

    def forward(self, images, captions, control_signal=None):
        # Returns: logits, cross_attention_weights
        raise NotImplementedError

    def generate(self, images, max_len=50, control_signal=None):
        """Autoregressive caption generation (greedy or beam search)."""
        raise NotImplementedError
```

In [ ]:
# ADD YOUR CODE HERE
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import math


class CNNEncoder(nn.Module):
    """CNN-based image encoder using pre-trained backbone."""
    def __init__(self, encoder_dim=2048, fine_tune_from=6):
        super(CNNEncoder, self).__init__()
        # Load pre-trained ResNet, remove final layers
        resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        modules = list(resnet.children())[:-2]
        self.resnet = nn.Sequential(*modules)

        # Enable fine-tuning for layers >= fine_tune_from
        for idx, child in enumerate(self.resnet.children()):
            if idx < fine_tune_from:
                for param in child.parameters():
                    param.requires_grad = False
            else:
                for param in child.parameters():
                    param.requires_grad = True

        self.encoder_dim = encoder_dim

    def forward(self, images):
        # Returns: (batch_size, num_patches, encoder_dim)
        features = self.resnet(images)              # (B, 2048, h, w)
        batch_size, channels, h, w = features.shape
        features = features.view(batch_size, channels, -1)  # (B, 2048, h*w)
        features = features.permute(0, 2, 1)                # (B, h*w, 2048)
        return features


class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding for Transformer."""
    def __init__(self, d_model, max_len=100, dropout=0.1):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


class TransformerDecoderLayer(nn.Module):
    """Single Transformer decoder layer with masked self-attention and cross-attention."""
    def __init__(self, d_model, num_heads, dim_feedforward, dropout=0.1):
        super(TransformerDecoderLayer, self).__init__()
        # Masked multi-head self-attention
        self.self_attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        # Cross-attention to encoder features
        self.cross_attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        # Feed-forward network
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, tgt, memory, tgt_mask=None, tgt_key_padding_mask=None):
        # Returns: output, cross_attention_weights (for visualization)

        # Masked self-attention + residual + norm
        tgt2, _ = self.self_attn(tgt, tgt, tgt, attn_mask=tgt_mask,
                                  key_padding_mask=tgt_key_padding_mask)
        tgt = self.norm1(tgt + self.dropout(tgt2))

        # Cross-attention + residual + norm
        tgt2, cross_attn_weights = self.cross_attn(tgt, memory, memory)
        tgt = self.norm2(tgt + self.dropout(tgt2))

        # Feed-forward + residual + norm
        tgt2 = self.linear2(self.dropout(F.relu(self.linear1(tgt))))
        tgt = self.norm3(tgt + self.dropout(tgt2))

        return tgt, cross_attn_weights


class TransformerDecoder(nn.Module):
    """Full Transformer decoder with embedding and output projection."""
    def __init__(self, vocab_size, d_model, num_heads, num_layers, dim_feedforward, dropout=0.1):
        super(TransformerDecoder, self).__init__()
        self.d_model = d_model

        # Word embedding + positional encoding
        self.word_embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model, dropout=dropout)

        # Control embedding (for controllable generation: 0=short, 1=medium, 2=long)
        self.control_embedding = nn.Embedding(3, d_model)

        # Stack of TransformerDecoderLayer
        self.layers = nn.ModuleList([
            TransformerDecoderLayer(d_model, num_heads, dim_feedforward, dropout)
            for _ in range(num_layers)
        ])

        # Output projection to vocab
        self.fc_out = nn.Linear(d_model, vocab_size)

    def generate_square_subsequent_mask(self, sz):
        """Generate causal mask for autoregressive decoding."""
        mask = (torch.triu(torch.ones(sz, sz)) == 1).transpose(0, 1)
        mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
        return mask

    def forward(self, captions, encoder_features, tgt_mask=None, control_signal=None):
        x = self.word_embedding(captions) * math.sqrt(self.d_model)

        # Add control embedding if provided
        if control_signal is not None:
            ctrl_emb = self.control_embedding(control_signal).unsqueeze(1)
            x = x + ctrl_emb

        x = self.pos_encoder(x)

        all_cross_attn_weights = []
        for layer in self.layers:
            x, cross_attn_weights = layer(x, encoder_features, tgt_mask=tgt_mask)
            all_cross_attn_weights.append(cross_attn_weights)

        logits = self.fc_out(x)
        return logits, all_cross_attn_weights


class ImageCaptioningModel(nn.Module):
    """Full image captioning model: Encoder + Transformer Decoder."""
    def __init__(self, vocab_size, encoder_dim=2048, d_model=512, num_heads=8, num_layers=3, dim_feedforward=2048):
        super(ImageCaptioningModel, self).__init__()
        self.encoder = CNNEncoder(encoder_dim=encoder_dim)
        self.encoder_projection = nn.Linear(encoder_dim, d_model)
        self.decoder = TransformerDecoder(vocab_size, d_model, num_heads, num_layers, dim_feedforward)

    def forward(self, images, captions, control_signal=None):
        # Returns: logits, cross_attention_weights
        encoder_features = self.encoder(images)
        encoder_features = self.encoder_projection(encoder_features)
        tgt_mask = self.decoder.generate_square_subsequent_mask(captions.size(1)).to(images.device)
        logits, cross_attn_weights = self.decoder(captions, encoder_features, tgt_mask, control_signal)
        return logits, cross_attn_weights

    def generate(self, images, max_len=50, control_signal=None, start_token_idx=1, end_token_idx=2):
        """Autoregressive caption generation (greedy)."""
        self.eval()
        with torch.no_grad():
            batch_size = images.size(0)
            encoder_features = self.encoder(images)
            encoder_features = self.encoder_projection(encoder_features)
            generated = torch.full((batch_size, 1), start_token_idx, dtype=torch.long, device=images.device)

            for _ in range(max_len):
                tgt_mask = self.decoder.generate_square_subsequent_mask(generated.size(1)).to(images.device)
                logits, attn_weights = self.decoder(generated, encoder_features, tgt_mask, control_signal)
                next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
                generated = torch.cat([generated, next_token], dim=1)
                if (next_token == end_token_idx).all():
                    break

            return generated, attn_weights[-1] if attn_weights else None

In [ ]:
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
from torch.utils.data import Subset

small_train_dataset = Subset(train_dataset, range(len(train_dataset) // 10))  # 10%
train_loader = DataLoader(small_train_dataset, batch_size=20, shuffle=True, collate_fn=collate_fn, num_workers=0)
val_loader   = DataLoader(val_dataset, batch_size=20, shuffle=False, collate_fn=collate_fn, num_workers=0)

print(f"Train size: {len(small_train_dataset)}")
print(f"Train batches: {len(train_loader)}")

In [ ]:
import gc
import os
from google.colab import drive
from torch.utils.data import Subset

drive.mount('/content/drive')

CHECKPOINT_PATH = '/content/drive/MyDrive/checkpoint_latest.pth'
BEST_MODEL_PATH = '/content/drive/MyDrive/best_model.pth'

# 10% data, batch 20
small_train_dataset = Subset(train_dataset, range(len(train_dataset) // 10))
train_loader = DataLoader(small_train_dataset, batch_size=20, shuffle=True, collate_fn=collate_fn, num_workers=0)
val_loader   = DataLoader(val_dataset, batch_size=20, shuffle=False, collate_fn=collate_fn, num_workers=0)
print(f"Train batches: {len(train_loader)}")

num_epochs    = 5
start_epoch   = 0
best_val_loss = float('inf')

model = ImageCaptioningModel(vocab_size=len(vocab.itos)).to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=1, factor=0.5)
scaler    = torch.cuda.amp.GradScaler() if device.type == 'cuda' else None

# Delete corrupted checkpoint if exists
if os.path.exists(CHECKPOINT_PATH):
    os.remove(CHECKPOINT_PATH)
    print("Deleted old checkpoint")

print("Starting from scratch.")

for epoch in range(start_epoch, num_epochs):
    model.train()
    total_loss = 0

    for i, (images, captions, lengths, masks) in enumerate(train_loader):
        images   = images.to(device)
        captions = captions.to(device)
        decoder_input = captions[:, :-1]
        targets       = captions[:, 1:]

        if scaler:
            with torch.cuda.amp.autocast():
                logits, _ = model(images, decoder_input)
                loss = criterion(logits.reshape(-1, logits.shape[-1]), targets.reshape(-1))
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
        else:
            logits, _ = model(images, decoder_input)
            loss = criterion(logits.reshape(-1, logits.shape[-1]), targets.reshape(-1))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad()

        if torch.isnan(loss):
            print(f"NaN detected at batch {i+1}, skipping...")
            optimizer.zero_grad()
            continue

        total_loss += loss.item()
        gc.collect()

        if (i + 1) % 20 == 0:
            print(f"Epoch {epoch+1}, Batch {i+1}, Loss: {loss.item():.4f}")
            torch.save({
                'epoch': epoch,
                'batch': i,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_val_loss': best_val_loss,
            }, CHECKPOINT_PATH)

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1} complete. Avg Loss: {avg_loss:.4f}")

    model.eval()
    val_loss = 0
    with torch.no_grad():
        for images, captions, lengths, masks in val_loader:
            images   = images.to(device)
            captions = captions.to(device)
            logits, _ = model(images, captions[:, :-1])
            val_loss += criterion(logits.reshape(-1, logits.shape[-1]), captions[:, 1:].reshape(-1)).item()
    val_loss /= len(val_loader)
    print(f"Validation Loss: {val_loss:.4f}")
    scheduler.step(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print(f"Saved best model with val_loss: {val_loss:.4f}")

    torch.save({
        'epoch': epoch + 1,
        'batch': -1,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'best_val_loss': best_val_loss,
    }, CHECKPOINT_PATH)
    print(f"Epoch {epoch+1} checkpoint saved to Drive.")

print("Training complete")

## **6. Defining Loss Function and Optimizer**

### **Loss Function**

For image captioning, we use **Cross-Entropy Loss** - the standard loss for sequence prediction tasks. It measures the difference between the predicted word probability distribution and the true word at each time step:

$$
\mathcal{L}_{CE} = -\sum_{t=1}^{T} \log P(y_t | y_{1:t-1}, X)
$$

where $T$ is the caption length and $y_t$ is the ground truth word at position $t$.

**Implementation notes:**
- Use `nn.CrossEntropyLoss(ignore_index=pad_idx)` to ignore padding tokens in the loss calculation
- During training, use **teacher forcing**: feed the ground truth tokens as input to predict the next token

```python
criterion = nn.CrossEntropyLoss(ignore_index=vocab['<pad>'])
```

---

### **Optimizer**

Common choices for training Transformers:

- **Adam/AdamW** - Recommended for most cases. AdamW adds proper weight decay.
- **SGD with momentum** - Can work but typically requires more tuning.

```python
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
```

Consider using a **learning rate scheduler** (e.g., `torch.optim.lr_scheduler.ReduceLROnPlateau`) to reduce the learning rate when validation loss plateaus.

**References:**
- [PyTorch Loss Functions](https://pytorch.org/docs/stable/nn.html#loss-functions)
- [PyTorch Optimizers](https://pytorch.org/docs/stable/optim.html)

In [ ]:
# ADD YOUR CODE HERE

import torch
import torch.nn as nn

# Loss Function
pad_idx = 0
criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)

# Optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)

# Learning Rate Scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

## **7. Training the Image Captioning Model**

Implement a training loop with the following features:

- Track **training loss** and **validation loss** per epoch
- Implement **early stopping** when validation loss stops improving
- **Save the best model** checkpoint based on validation loss
- Store losses for later visualization

```python
def train_epoch(model, dataloader, criterion, optimizer, device):
    # Returns: average training loss
    raise NotImplementedError


def validate(model, dataloader, criterion, device):
    # Returns: average validation loss
    raise NotImplementedError


def train(model, train_loader, val_loader, criterion, optimizer, num_epochs, device, patience=5):
    # Returns: train_losses, val_losses
    raise NotImplementedError
```

In [ ]:
# ADD YOUR CODE HERE

import torch
import copy

def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0

    for images, captions, lengths, masks in dataloader:
        # Move to device
        images = images.to(device)
        captions = captions.to(device)

        # Teacher forcing: shift captions
        decoder_input = captions[:, :-1]
        targets = captions[:, 1:]

        # Forward pass
        logits, _ = model(images, decoder_input)

        # Compute loss
        logits = logits.reshape(-1, logits.shape[-1])
        targets = targets.reshape(-1)
        loss = criterion(logits, targets)

        # Backprop
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)


def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for images, captions, lengths, masks in dataloader:
            images = images.to(device)
            captions = captions.to(device)

            decoder_input = captions[:, :-1]
            targets = captions[:, 1:]

            logits, _ = model(images, decoder_input)

            logits = logits.reshape(-1, logits.shape[-1])
            targets = targets.reshape(-1)
            loss = criterion(logits, targets)

            total_loss += loss.item()

    return total_loss / len(dataloader)


def train(model, train_loader, val_loader, criterion, optimizer, num_epochs, device, patience=5):
    train_losses = []
    val_losses = []
    best_val_loss = float('inf')
    best_model_state = None
    epochs_no_improve = 0

    for epoch in range(num_epochs):
        # Train
        train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
        train_losses.append(train_loss)

        # Validate
        val_loss = validate(model, val_loader, criterion, device)
        val_losses.append(val_loss)

        print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
            print(f"  -> New best model! Saving checkpoint...")
        else:
            epochs_no_improve += 1
            print(f"  -> No improvement for {epochs_no_improve} epochs")

            if epochs_no_improve >= patience:
                print(f"Early stopping triggered after {epoch+1} epochs")
                break

    # Load best model
    model.load_state_dict(best_model_state)
    torch.save(best_model_state, 'best_model.pth')
    print(f"Best model saved to 'best_model.pth' (Val Loss: {best_val_loss:.4f})")

    return train_losses, val_losses


## **8.1 Visualizing Training Metrics**

- **Restore the model's parameters** from the checkpoint where validation loss was lowest to use the most optimal version of the model.
- Use `Matplotlib` or `Seaborn` to plot the loss curves over epochs.

Did your model **converge**? Explain your results!

```python
def plot_losses(train_losses, val_losses):
    raise NotImplementedError
```

In [ ]:
# ADD YOUR CODE HERE
import matplotlib.pyplot as plt
import torch

def plot_losses(train_losses, val_losses):
    epochs = range(1, len(train_losses) + 1)
    plt.figure(figsize=(10, 6))
    plt.plot(epochs, train_losses, label='Train Loss')
    plt.plot(epochs, val_losses, label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training and Validation Loss Curves')
    plt.legend()
    plt.grid(True)
    plt.show()

# Restore best model
model.load_state_dict(torch.load('best_model.pth', weights_only=True))
print("Best model loaded")

train_losses = [3.6963, 2.9588, 2.6799, 2.4694, 2.2957]
val_losses   = [3.1850, 2.9906, 2.9232, 2.9113, 2.9334]

plot_losses(train_losses, val_losses)

print("Train loss steadily drops (3.70 → 2.30), but val loss plateaus around 2.91 from epoch 3 onward — mild overfitting starting. Best val loss was epoch 4 (2.9113).")

## **8.2 Visualizing Cross-Attention Weights**

Visualize the **cross-attention weights** from your Transformer decoder to understand what image regions the model focuses on when generating each word.

For each generated word, the cross-attention layer produces weights over the image patches. By reshaping these weights back to the spatial dimensions and overlaying them on the original image, you can create interpretable **attention heatmaps**.

- Show attention maps for at least **5 different images** from the validation set
- For each image, display the attention heatmap for **each word** in the generated caption
- Optionally, average attention across decoder layers or heads for clearer visualization

```python
def visualize_cross_attention(image, caption_tokens, attention_weights):
    raise NotImplementedError
```

In [ ]:
# ADD YOUR CODE HERE

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image

def visualize_cross_attention(image_tensor, caption_tokens, attention_weights, vocab, save_path=None):
    words = []
    for tok in caption_tokens:
        if isinstance(tok, torch.Tensor):
            tok = tok.item()
        if tok in [0, 1, 2]:
            continue
        words.append(vocab.itos.get(tok, '<unk>'))

    # Denormalize image
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])
    img  = image_tensor.cpu().permute(1, 2, 0).numpy()
    img  = np.clip(std * img + mean, 0, 1)

    attn       = attention_weights.cpu().detach().float().numpy()
    num_words  = min(len(words), attn.shape[0])
    patch_size = int(attn.shape[1] ** 0.5)  # 7 for ResNet50

    cols = min(num_words + 1, 6)
    rows = (num_words + 1 + cols - 1) // cols + 1
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
    axes = axes.flatten()

    axes[0].imshow(img)
    axes[0].set_title("Original", fontsize=9)
    axes[0].axis('off')

    for w_idx in range(num_words):
        ax      = axes[w_idx + 1]
        attn_map = attn[w_idx].reshape(patch_size, patch_size)
        attn_map = (attn_map - attn_map.min()) / (attn_map.max() - attn_map.min() + 1e-8)

        attn_img = Image.fromarray((attn_map * 255).astype(np.uint8))
        attn_img = attn_img.resize((img.shape[1], img.shape[0]), Image.BILINEAR)
        attn_img = np.array(attn_img) / 255.0

        ax.imshow(img)
        ax.imshow(attn_img, alpha=0.5, cmap='jet')
        ax.set_title(words[w_idx], fontsize=9)
        ax.axis('off')

    for idx in range(num_words + 1, len(axes)):
        axes[idx].axis('off')

    plt.suptitle("Cross-Attention Heatmaps", fontsize=12)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved to {save_path}")
    plt.show()


def show_attention_for_validation_images(model, val_dataset, vocab, device, num_images=5):
    model.eval()
    indices = np.random.choice(len(val_dataset), num_images, replace=False)

    for idx in indices:
        # ← handles datasets returning 3 or 4 items
        sample       = val_dataset[idx]
        image_tensor = sample[0]
        image_input  = image_tensor.unsqueeze(0).to(device)

        with torch.no_grad():
            generated, attn_weights = model.generate(
                image_input,
                start_token_idx=1,
                end_token_idx=2
            )

        if attn_weights is None:
            print(f"Image {idx}: No attention weights returned, skipping.")
            continue

        # generated: (1, seq_len) | attn_weights: (1, tgt_len, src_len)
        caption_tokens = generated[0].cpu().tolist()
        attn           = attn_weights[0]  # (tgt_len, src_len)

        words = [vocab.itos.get(t, '<unk>') for t in caption_tokens if t not in [0, 1, 2]]
        print(f"\nImage {idx} → {' '.join(words)}")

        visualize_cross_attention(
            image_tensor     = image_tensor,
            caption_tokens   = caption_tokens,
            attention_weights= attn,
            vocab            = vocab,
            save_path        = f'/content/drive/MyDrive/attention_image_{idx}.png'
        )


show_attention_for_validation_images(model, val_dataset, vocab, device, num_images=5)

## **8.3 Running Inference on the Image Captioning Model**

Pass validation images through the **custom-trained image captioning model** and evaluate its performance. Use the **BLEU score** as the evaluation metric, computing **BLEU-1, BLEU-2, BLEU-3, and BLEU-4**.

**Evaluation Requirements:**
- Compute BLEU scores using **all 5 reference captions** for each image
- Report average BLEU scores across the entire validation set
- Show generated captions for at least 10 sample images alongside ground-truth captions

**Decoding Options:**
- **Greedy decoding** (required): Select the most probable word at each step
- **Beam search** (extra credit): Maintain top-k candidate sequences

**Also demonstrate your controllable generation:**
- Show how different control signals (length/style/focus) affect the generated captions for the same image

```python
def generate_caption(model, image, vocab, max_len=50, control_signal=None):
    # Returns: caption string, attention_weights
    raise NotImplementedError


def evaluate_model(model, dataloader, vocab):
    # Returns: dict with BLEU-1, BLEU-2, BLEU-3, BLEU-4
    raise NotImplementedError
```

In [ ]:
# ADD YOUR CODE HERE
import torch
import nltk
import numpy as np
import matplotlib.pyplot as plt
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

nltk.download('punkt', quiet=True)


def generate_caption(model, image, vocab, max_len=50, control_signal=None):
    model.eval()
    with torch.no_grad():
        if image.dim() == 3:
            image = image.unsqueeze(0)
        image = image.to(device)

        if control_signal is not None:
            control_signal = torch.tensor([control_signal], dtype=torch.long, device=device)

        generated, attn_weights = model.generate(
            image,
            max_len=max_len,
            control_signal=control_signal,
            start_token_idx=vocab.stoi['<sos>'],
            end_token_idx=vocab.stoi['<eos>']
        )

        tokens  = generated[0].cpu().tolist()
        words   = [vocab.itos[t] for t in tokens if t not in [
            vocab.stoi['<pad>'], vocab.stoi['<sos>'], vocab.stoi['<eos>']
        ]]
        caption = ' '.join(words)

    return caption, attn_weights


def evaluate_model(model, val_dataset, vocab, num_samples=500):
    model.eval()
    smoothie = SmoothingFunction().method1

    references_corpus = []
    hypotheses_corpus = []
    indices = list(range(min(num_samples, len(val_dataset))))

    print(f"Evaluating BLEU on {len(indices)} validation images...")

    for idx in indices:
        # ← safe unpacking
        image_tensor = val_dataset[idx][0]
        img_id       = val_dataset.img_ids[idx]

        ann_ids = val_dataset.coco.getAnnIds(imgIds=img_id)
        anns    = val_dataset.coco.loadAnns(ann_ids)
        refs    = [nltk.word_tokenize(ann['caption'].lower()) for ann in anns]

        caption, _ = generate_caption(model, image_tensor, vocab, max_len=50)
        hyp        = nltk.word_tokenize(caption.lower())

        references_corpus.append(refs)
        hypotheses_corpus.append(hyp)

    bleu1 = corpus_bleu(references_corpus, hypotheses_corpus, weights=(1, 0, 0, 0))
    bleu2 = corpus_bleu(references_corpus, hypotheses_corpus, weights=(0.5, 0.5, 0, 0))
    bleu3 = corpus_bleu(references_corpus, hypotheses_corpus, weights=(0.33, 0.33, 0.33, 0))
    bleu4 = corpus_bleu(references_corpus, hypotheses_corpus, weights=(0.25, 0.25, 0.25, 0.25))

    results = {
        'BLEU-1': round(bleu1 * 100, 2),
        'BLEU-2': round(bleu2 * 100, 2),
        'BLEU-3': round(bleu3 * 100, 2),
        'BLEU-4': round(bleu4 * 100, 2),
    }

    print("\nBLEU Scores")
    for k, v in results.items():
        print(f"  {k}: {v}")

    return results


def show_sample_captions(model, val_dataset, vocab, num_samples=10):
    indices = np.random.choice(len(val_dataset), num_samples, replace=False)
    mean    = np.array([0.485, 0.456, 0.406])
    std     = np.array([0.229, 0.224, 0.225])

    for i, idx in enumerate(indices):
        # ← safe unpacking
        image_tensor = val_dataset[idx][0]
        img_id       = val_dataset.img_ids[idx]

        ann_ids = val_dataset.coco.getAnnIds(imgIds=img_id)
        anns    = val_dataset.coco.loadAnns(ann_ids)
        gt_caps = [ann['caption'] for ann in anns]

        caption, _ = generate_caption(model, image_tensor, vocab)

        img = image_tensor.permute(1, 2, 0).numpy()
        img = np.clip(std * img + mean, 0, 1)

        plt.figure(figsize=(8, 4))
        plt.imshow(img)
        plt.axis('off')
        plt.title(f"Generated: {caption}\n\nGT: {gt_caps[0]}", fontsize=9, wrap=True)
        plt.tight_layout()
        plt.savefig(f'/content/drive/MyDrive/sample_{i+1}.png', dpi=100, bbox_inches='tight')
        plt.show()

        print(f"\n Sample {i+1}")
        print(f"Generated : {caption}")
        for j, gt in enumerate(gt_caps):
            print(f"Reference {j+1}: {gt}")


def demo_controllable_generation(model, val_dataset, vocab, num_images=3):
    control_labels = {0: 'Short', 1: 'Medium', 2: 'Long'}
    indices        = np.random.choice(len(val_dataset), num_images, replace=False)

    print("\nControllable Generation Demo")
    for idx in indices:
        # ← safe unpacking
        image_tensor = val_dataset[idx][0]
        img_id       = val_dataset.img_ids[idx]

        ann_ids = val_dataset.coco.getAnnIds(imgIds=img_id)
        anns    = val_dataset.coco.loadAnns(ann_ids)
        gt      = anns[0]['caption']

        print(f"\nImage ID: {img_id}")
        print(f"Ground Truth: {gt}")

        for ctrl_val, ctrl_name in control_labels.items():
            caption, _ = generate_caption(model, image_tensor, vocab, control_signal=ctrl_val)
            print(f"  Control={ctrl_name}: {caption}")

bleu_results = evaluate_model(model, val_dataset, vocab, num_samples=500)
show_sample_captions(model, val_dataset, vocab, num_samples=10)
demo_controllable_generation(model, val_dataset, vocab, num_images=3)

## **9. Loading an Existing Image Captioning Model**

Load a pre-trained image captioning model for comparison. Recommended options:

- **BLIP:** Available on [Hugging Face](https://huggingface.co/Salesforce/blip-image-captioning-base)
- **ViT-GPT2:** Available on [Hugging Face](https://huggingface.co/nlpconnect/vit-gpt2-image-captioning)

```python
def load_pretrained_model():
    # Returns: model, processor/tokenizer
    raise NotImplementedError
```

In [ ]:
# ADD YOUR CODE HERE
from transformers import BlipProcessor, BlipForConditionalGeneration

def load_pretrained_model():
    processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
    model     = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")
    model     = model.to(device)
    model.eval()
    return model, processor

blip_model, processor = load_pretrained_model()
print("BLIP loaded.")

## **10. Evaluating the Existing Image Captioning Model**

Apply the **same metrics** used for your custom model to ensure a fair comparison.

```python
def evaluate_pretrained_model(model, processor, dataloader):
    # Returns: dict with BLEU-1, BLEU-2, BLEU-3, BLEU-4
    raise NotImplementedError
```

In [ ]:
# ADD YOUR CODE HERE

def evaluate_pretrained_model(blip_model, processor, val_dataset, num_samples=500):
    from nltk.translate.bleu_score import corpus_bleu
    from torchvision.transforms.functional import to_pil_image
    import nltk

    blip_model.eval()
    references_corpus = []
    hypotheses_corpus = []
    indices = list(range(min(num_samples, len(val_dataset))))

    print(f"Evaluating BLIP on {len(indices)} validation images...")

    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    for idx in indices:
        image_tensor = val_dataset[idx][0]
        img_id       = val_dataset.img_ids[idx]

        # Denormalize and convert to PIL
        image     = (image_tensor.cpu() * std + mean).clamp(0, 1)
        pil_image = to_pil_image(image)

        # Generate caption
        inputs  = processor(images=pil_image, return_tensors="pt").to(device)
        with torch.no_grad():
            out = blip_model.generate(**inputs, max_new_tokens=50)
        caption = processor.decode(out[0], skip_special_tokens=True)

        # Ground truth references
        ann_ids = val_dataset.coco.getAnnIds(imgIds=img_id)
        anns    = val_dataset.coco.loadAnns(ann_ids)
        refs    = [nltk.word_tokenize(ann['caption'].lower()) for ann in anns]
        hyp     = nltk.word_tokenize(caption.lower())

        references_corpus.append(refs)
        hypotheses_corpus.append(hyp)

    bleu1 = corpus_bleu(references_corpus, hypotheses_corpus, weights=(1, 0, 0, 0))
    bleu2 = corpus_bleu(references_corpus, hypotheses_corpus, weights=(0.5, 0.5, 0, 0))
    bleu3 = corpus_bleu(references_corpus, hypotheses_corpus, weights=(0.33, 0.33, 0.33, 0))
    bleu4 = corpus_bleu(references_corpus, hypotheses_corpus, weights=(0.25, 0.25, 0.25, 0.25))

    return {
        'BLEU-1': round(bleu1 * 100, 2),
        'BLEU-2': round(bleu2 * 100, 2),
        'BLEU-3': round(bleu3 * 100, 2),
        'BLEU-4': round(bleu4 * 100, 2),
    }

blip_results = evaluate_pretrained_model(blip_model, processor, val_dataset, num_samples=500)
print(blip_results)

## **11. Comparing the Two Models**

Compare the performance of both models using **BLEU-1, BLEU-2, BLEU-3, and BLEU-4** scores. Visualize predictions from both models on the **same validation images**.

**Requirements:**
- Create a comparison table with BLEU scores for both models
- Show side-by-side caption comparisons for at least 10 images
- Compare attention/focus patterns if the pre-trained model provides attention weights

```python
def compare_models(custom_results, pretrained_results):
    # Display comparison table and visualizations
    raise NotImplementedError
```

**Analysis (Written Response Required):**
1. Why does the pre-trained model perform better/worse on certain images?
2. What architectural differences contribute to performance gaps?
3. How does your Transformer decoder compare to the decoder used in the pre-trained model?
4. Propose **at least 3 specific improvements** for your custom model based on this analysis.

> **Answer**:


1. Why does the pre-trained model perform better/ worse on certain images?

BLIP was trained on 130 million image-text pairs from the internet. So for common objects like people, dogs, cars — it performs well because it has seen millions of similar images. My model was trained on only ~10% of COCO for 5 epochs, so it struggles with complex scenes or rare objects. BLIP also does worse on very specific or unusual images because its captions tend to be generic.


My model prediction is not getting correclty prediciting as close as to picture as well as in term of image capationing.

2. What architectural differences contribute to performance gaps?

BLIP uses a vision transformer (ViT) as encoder — it splits the image into patches and processes them with full attention. My model uses ResNet50 which extracts CNN features in a grid. ViT captures global context better. BLIP also uses a BERT-based decoder pretrained on language, so it generates more natural sentences. My decoder was trained from scratch with random weights.


3. How does your Transformer decoder compare to the decoder used in the pre-trained model?

My decoder is a standard Transformer with cross-attention to ResNet features built from scratch with 47M parameters trained only on COCO captions. BLIP's decoder is based on BERT, already pretrained on massive text data, so it understands language much better before even seeing images. My decoder has to learn both language and vision alignment at the same time, which needs more data and epochs.


4. Three specific improvements:

1. Replace ResNet50 with ViT encoder —patch-based attention captures spatial relationships better than CNN grids.

2. Pretrain the decoder on text initialize with a pretrained language model like GPT2 so it already understands grammar before training on captions.

3. Train on full COCO I used 10% of the data due to hardware limits; more data would directly improve BLEU scores across all metrics

In [ ]:
# ADD YOUR CODE HERE

def compare_models(custom_results, pretrained_results, model, blip_model, processor, val_dataset, vocab, num_images=10):
    from torchvision.transforms.functional import to_pil_image
    import matplotlib.pyplot as plt
    import numpy as np

    mean   = np.array([0.485, 0.456, 0.406])
    std    = np.array([0.229, 0.224, 0.225])
    mean_t = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    std_t  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)

    # Bleu Comparision table
    print("\n" + "="*50)
    print(f"{'Metric':<12} {'Model':>15} {'BLIP':>15}")
    print("="*50)
    for k in custom_results:
        print(f"{k:<12} {custom_results[k]:>14.2f}  {pretrained_results[k]:>14.2f}")
    print("="*50)

    # Bleu bar chart
    metrics      = list(custom_results.keys())
    model_scores = list(custom_results.values())
    blip_scores  = list(pretrained_results.values())
    x     = np.arange(len(metrics))
    width = 0.35

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(x - width/2, model_scores, width, label='Model')
    ax.bar(x + width/2, blip_scores,  width, label='BLIP')
    ax.set_ylabel('BLEU Score')
    ax.set_title('BLEU Score Comparison')
    ax.set_xticks(x)
    ax.set_xticklabels(metrics)
    ax.legend()
    ax.grid(axis='y')
    plt.tight_layout()
    plt.savefig('/content/drive/MyDrive/bleu_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

    # side by side comparison
    indices = np.random.choice(len(val_dataset), num_images, replace=False)

    for i, idx in enumerate(indices):
        image_tensor = val_dataset[idx][0]
        img_id       = val_dataset.img_ids[idx]

        # Ground truth
        ann_ids = val_dataset.coco.getAnnIds(imgIds=img_id)
        anns    = val_dataset.coco.loadAnns(ann_ids)
        gt      = anns[0]['caption']

        # Model caption
        model_caption, _ = generate_caption(model, image_tensor, vocab)

        # BLIP caption
        pil_image = to_pil_image((image_tensor.cpu() * std_t + mean_t).clamp(0, 1))
        inputs    = processor(images=pil_image, return_tensors="pt").to(device)
        with torch.no_grad():
            out = blip_model.generate(**inputs, max_new_tokens=50)
        blip_caption = processor.decode(out[0], skip_special_tokens=True)

        # Denormalize image
        img = image_tensor.permute(1, 2, 0).numpy()
        img = np.clip(std * img + mean, 0, 1)

        # Plot side by side
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))

        axes[0].imshow(img)
        axes[0].axis('off')
        axes[0].set_title(f"Model:\n{model_caption}", fontsize=8, wrap=True)

        axes[1].imshow(img)
        axes[1].axis('off')
        axes[1].set_title(f"BLIP:\n{blip_caption}", fontsize=8, wrap=True)

        plt.suptitle(f"GT: {gt}", fontsize=9, wrap=True)
        plt.tight_layout()
        plt.savefig(f'/content/drive/MyDrive/compare_{i+1}.png', dpi=100, bbox_inches='tight')
        plt.show()

        print(f"\n Image {i+1} ")
        print(f"GT    : {gt}")
        print(f"Model : {model_caption}")
        print(f"BLIP  : {blip_caption}")

compare_models(bleu_results, blip_results, model, blip_model, processor, val_dataset, vocab, num_images=10)

---
**Remember to save and upload your `.ipynb` file to Canvas before the deadline!**